# 01 - Data Exploration
**Vehicle Predictive Maintenance Project**

This notebook explores all five raw data sources, documents structure, identifies data quality issues, and flags decisions made ahead of cleaning.

**Data Sources:**
- `Vehicle_Info.xlsx` — registration, make, model, asset type, status
- `Vehicle_Driver.xlsx` — current driver assignment, MOT dates
- `Vehicle_Mileage.xlsx` — monthly mileage per vehicle
- `Maintenance_Records.xlsx` — repair history with category and cost
- `Driver_Scores.xlsx` — weekly driver behaviour scores

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13

RAW = '../data/raw/'
print('Setup complete.')

## 2. Load All Datasets

In [ ]:
vehicle_info    = pd.read_excel(RAW + 'Vehicle_Info.xlsx')
vehicle_driver  = pd.read_excel(RAW + 'Vehicle_Driver.xlsx')
vehicle_mileage = pd.read_excel(RAW + 'Vehicle_Mileage.xlsx')
maintenance     = pd.read_excel(RAW + 'Maintenance_Records.xlsx')
driver_scores   = pd.read_excel(RAW + 'Driver_Scores.xlsx')

datasets = {
    'Vehicle_Info': vehicle_info,
    'Vehicle_Driver': vehicle_driver,
    'Vehicle_Mileage': vehicle_mileage,
    'Maintenance_Records': maintenance,
    'Driver_Scores': driver_scores
}

print('All datasets loaded successfully.\n')
for name, df in datasets.items():
    print(f'{name:25s} — {df.shape[0]:>6,} rows  x  {df.shape[1]:>2} cols')

## 3. Dataset Summaries

In [ ]:
def dataset_summary(df, name):
    print(f"{'='*65}")
    print(f" {name}")
    print(f"{'='*65}")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")
    summary = pd.DataFrame({
        'dtype': df.dtypes,
        'missing': df.isnull().sum(),
        'missing_%': (df.isnull().sum() / len(df) * 100).round(1),
        'unique': df.nunique()
    })
    print(summary)
    print(f"\nFirst 3 rows:")
    display(df.head(3))
    print()

for name, df in datasets.items():
    dataset_summary(df, name)

## 4. Vehicle Info — Exploration

In [ ]:
print('Asset Type breakdown:')
print(vehicle_info['Asset Type'].value_counts())
print()
print('Vehicle Status breakdown:')
print(vehicle_info['Vehicle Status'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vehicle_info['Asset Type'].value_counts().plot(kind='bar', ax=axes[0], color=sns.color_palette('muted'))
axes[0].set_title('Vehicle Count by Asset Type')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

vehicle_info['Vehicle Status'].value_counts().plot(kind='bar', ax=axes[1], color=sns.color_palette('muted'))
axes[1].set_title('Vehicle Count by Status')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../outputs/reports/01_vehicle_info_breakdown.png', dpi=150)
plt.show()

In [ ]:
# Filter to vans only (exclude Cars and any junk values)
van_types = ['Medium Van', 'Small Van', 'Large Van']
vans_only = vehicle_info[vehicle_info['Asset Type'].isin(van_types)]
print(f'Total vehicles in Vehicle_Info: {len(vehicle_info):,}')
print(f'Vans only: {len(vans_only):,}')
print(f'Cars / other excluded: {len(vehicle_info) - len(vans_only):,}')
print()
print('Van status breakdown:')
print(vans_only['Vehicle Status'].value_counts())

In [ ]:
# Vehicle age distribution (vans only)
vans_only = vans_only.copy()
vans_only['Registered Date'] = pd.to_datetime(vans_only['Registered Date'])
vans_only['Vehicle Age (Years)'] = (pd.Timestamp.now() - vans_only['Registered Date']).dt.days / 365.25

plt.figure(figsize=(12, 4))
sns.histplot(vans_only['Vehicle Age (Years)'], bins=30, kde=True)
plt.title('Distribution of Van Ages (Years)')
plt.xlabel('Age (Years)')
plt.tight_layout()
plt.savefig('../outputs/reports/01_vehicle_age_distribution.png', dpi=150)
plt.show()

print(vans_only['Vehicle Age (Years)'].describe())

In [ ]:
# Make breakdown for vans
plt.figure(figsize=(12, 4))
vans_only['Make'].value_counts().head(15).plot(kind='bar')
plt.title('Top 15 Van Makes')
plt.xlabel('')
plt.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../outputs/reports/01_van_makes.png', dpi=150)
plt.show()

## 5. Vehicle Driver — Exploration

In [ ]:
print('Asset Type in Vehicle_Driver:')
print(vehicle_driver['Asset Type'].value_counts())
print()
print('Vehicle Status in Vehicle_Driver:')
print(vehicle_driver['Vehicle Status'].value_counts())

In [ ]:
# Missing driver names
missing_driver = vehicle_driver['Driver'].isnull().sum()
total = len(vehicle_driver)
print(f'Missing driver assignments: {missing_driver} ({missing_driver/total*100:.1f}%)')
print('\n>> Decision: Vehicles with missing drivers will receive the fleet average driver score.')

# Missing branches
missing_branch = vehicle_driver['Branch'].isnull().sum()
print(f'\nMissing branch: {missing_branch} ({missing_branch/total*100:.1f}%)')
print('>> Decision: Missing branches will be filled from Vehicle_Info using registration number.')

In [ ]:
# MOT date exploration
vehicle_driver['Next MOT Date'] = pd.to_datetime(vehicle_driver['Next MOT Date'], errors='coerce')
today = pd.Timestamp.now()

vehicle_driver['Days Until MOT'] = (vehicle_driver['Next MOT Date'] - today).dt.days

print('MOT Date summary:')
print(vehicle_driver['Next MOT Date'].describe())
print()

# How many have MOT due within 90 days
due_soon = vehicle_driver[vehicle_driver['Days Until MOT'].between(0, 90)]
overdue  = vehicle_driver[vehicle_driver['Days Until MOT'] < 0]
print(f'MOT due within 90 days: {len(due_soon)}')
print(f'MOT overdue (past date): {len(overdue)}')

In [ ]:
# MOT distribution
mot_data = vehicle_driver['Days Until MOT'].dropna()
mot_data = mot_data[mot_data.between(-30, 400)]  # Reasonable window

plt.figure(figsize=(12, 4))
sns.histplot(mot_data, bins=40, kde=False)
plt.axvline(0, color='red', linestyle='--', label='Today')
plt.axvline(90, color='orange', linestyle='--', label='90 Day Warning')
plt.title('Days Until Next MOT')
plt.xlabel('Days')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/reports/01_mot_dates.png', dpi=150)
plt.show()

## 6. Vehicle Mileage — Exploration

In [ ]:
print('Mileage data types:')
print(vehicle_mileage.dtypes)
print()
print('Sample:')
display(vehicle_mileage.head(10))

# Check for non-numeric values in Average Miles Per Day
non_numeric = pd.to_numeric(vehicle_mileage['Average Miles Per Day'], errors='coerce').isnull().sum()
print(f'\nNon-numeric values in Average Miles Per Day: {non_numeric}')

non_numeric_year = pd.to_numeric(vehicle_mileage['Year'], errors='coerce').isnull().sum()
print(f'Non-numeric values in Year: {non_numeric_year}')

# Show problematic values
problem_rows = vehicle_mileage[pd.to_numeric(vehicle_mileage['Average Miles Per Day'], errors='coerce').isnull()]
if len(problem_rows) > 0:
    print('\nProblematic Average Miles Per Day values:')
    print(problem_rows['Average Miles Per Day'].value_counts().head(20))

In [ ]:
# Convert and clean mileage
vehicle_mileage['Average Miles Per Day'] = pd.to_numeric(vehicle_mileage['Average Miles Per Day'], errors='coerce')
vehicle_mileage['Year'] = pd.to_numeric(vehicle_mileage['Year'], errors='coerce')

print('After conversion:')
print(f'Missing Average Miles Per Day: {vehicle_mileage["Average Miles Per Day"].isnull().sum()}')
print(f'Missing Year: {vehicle_mileage["Year"].isnull().sum()}')
print()
print('Year range:', vehicle_mileage['Year'].min(), '-', vehicle_mileage['Year'].max())
print('Months:', vehicle_mileage['Month'].unique())

In [ ]:
# Monthly mileage trend across fleet
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
vehicle_mileage['Month'] = pd.Categorical(vehicle_mileage['Month'], categories=month_order, ordered=True)

monthly_avg = vehicle_mileage.groupby(['Year', 'Month'])['Average Miles Per Day'].mean().reset_index()
monthly_avg = monthly_avg.sort_values(['Year', 'Month'])
monthly_avg['Period'] = monthly_avg['Year'].astype(str) + '-' + monthly_avg['Month'].astype(str)

plt.figure(figsize=(16, 4))
plt.plot(monthly_avg['Period'], monthly_avg['Average Miles Per Day'], marker='o', markersize=3)
plt.title('Fleet Average Miles Per Day by Month')
plt.xlabel('')
plt.ylabel('Avg Miles/Day')
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig('../outputs/reports/01_monthly_mileage_trend.png', dpi=150)
plt.show()

## 7. Maintenance Records — Exploration

In [ ]:
print('Date columns — checking formats:')
maintenance['Invoice Date'] = pd.to_datetime(maintenance['Invoice Date'], errors='coerce')
maintenance['Booking Date'] = pd.to_datetime(maintenance['Booking Date'], errors='coerce')
maintenance['Incident Start Date'] = pd.to_datetime(maintenance['Incident Start Date'], errors='coerce')

print(f'Invoice Date range: {maintenance["Invoice Date"].min()} to {maintenance["Invoice Date"].max()}')
print(f'Missing Invoice Date: {maintenance["Invoice Date"].isnull().sum()}')
print(f'Missing Net Amount: {maintenance["Net Amount"].isnull().sum()}')

In [ ]:
# Top Category breakdown
print('Top Category breakdown:')
print(maintenance['Top Category'].value_counts())

In [ ]:
# Cost distribution
print('Net Amount summary:')
print(maintenance['Net Amount'].describe())
print()

# Flag negative or zero amounts
neg = maintenance[maintenance['Net Amount'] <= 0]
print(f'Zero or negative Net Amount records: {len(neg)}')
if len(neg) > 0:
    print(neg['Net Amount'].value_counts().head(10))

In [ ]:
# Repairs over time
maintenance['Year-Month'] = maintenance['Invoice Date'].dt.to_period('M')
repairs_over_time = maintenance.groupby('Year-Month').size().reset_index(name='count')
repairs_over_time['Year-Month'] = repairs_over_time['Year-Month'].astype(str)

plt.figure(figsize=(16, 4))
plt.bar(repairs_over_time['Year-Month'], repairs_over_time['count'], color='steelblue')
plt.title('Number of Maintenance Records by Month')
plt.xlabel('')
plt.ylabel('Count')
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig('../outputs/reports/01_repairs_over_time.png', dpi=150)
plt.show()

In [ ]:
# Repairs per vehicle
repairs_per_vehicle = maintenance.groupby('Registration Number').size()
print('Repairs per vehicle summary:')
print(repairs_per_vehicle.describe())

plt.figure(figsize=(12, 4))
sns.histplot(repairs_per_vehicle, bins=40, kde=False)
plt.title('Distribution of Repairs per Vehicle')
plt.xlabel('Number of Repairs')
plt.tight_layout()
plt.savefig('../outputs/reports/01_repairs_per_vehicle.png', dpi=150)
plt.show()

In [ ]:
# Flag: Environmental Disposal — likely not a mechanical repair
env_disposal = maintenance[maintenance['category'].str.contains('Environmental Disposal', na=False)]
print(f'Environmental Disposal records: {len(env_disposal)}')
print('>> Note: These are likely waste disposal charges, not mechanical repairs.')
print('>> Decision: Will be excluded from predictive modelling in cleaning notebook.')

## 8. Driver Scores — Exploration

In [ ]:
print('Driver Scores shape:', driver_scores.shape)
print('Date range:', driver_scores['Week'].min(), 'to', driver_scores['Week'].max())
print('Unique drivers:', driver_scores['DriverName'].nunique())
print()
print('DriverScore summary:')
print(driver_scores['DriverScore'].describe())

In [ ]:
# Overall driver score distribution
plt.figure(figsize=(12, 4))
sns.histplot(driver_scores['DriverScore'], bins=50, kde=True)
plt.title('Distribution of Weekly Driver Scores')
plt.xlabel('Driver Score')
plt.tight_layout()
plt.savefig('../outputs/reports/01_driver_score_distribution.png', dpi=150)
plt.show()

In [ ]:
# Average score per driver (what we'll use as the joined score)
avg_driver_score = driver_scores.groupby('DriverName')['DriverScore'].mean().reset_index()
avg_driver_score.columns = ['DriverName', 'AvgDriverScore']

print('Average Driver Score per driver summary:')
print(avg_driver_score['AvgDriverScore'].describe())

fleet_avg_score = avg_driver_score['AvgDriverScore'].mean()
print(f'\nFleet average score (used for missing driver assignments): {fleet_avg_score:.1f}')

In [ ]:
# Sub-score breakdown
score_cols = ['DeccelerationRate', 'AccelerationRate', 'SpeedingRate', 'CorneringRate']

# Convert to numeric
for col in score_cols:
    driver_scores[col] = pd.to_numeric(driver_scores[col], errors='coerce')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, col in enumerate(score_cols):
    sns.histplot(driver_scores[col].dropna(), bins=30, ax=axes[i], kde=True)
    axes[i].set_title(col)
    axes[i].set_xlabel('')

plt.suptitle('Driver Sub-Score Distributions', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/reports/01_driver_subscores.png', dpi=150)
plt.show()

In [ ]:
# Driver score trend over time (fleet average)
weekly_avg = driver_scores.groupby('Week')['DriverScore'].mean().reset_index()

plt.figure(figsize=(14, 4))
plt.plot(weekly_avg['Week'], weekly_avg['DriverScore'], linewidth=1)
plt.title('Fleet Average Driver Score Over Time (Weekly)')
plt.xlabel('Week')
plt.ylabel('Average Score')
plt.tight_layout()
plt.savefig('../outputs/reports/01_driver_score_trend.png', dpi=150)
plt.show()

## 9. Cross-Dataset Linkage Check

In [ ]:
# How well do the registration numbers link across datasets?
regs_info      = set(vehicle_info['RegistrationNumber'].dropna())
regs_driver    = set(vehicle_driver['RegistrationNumber'].dropna())
regs_mileage   = set(vehicle_mileage['RegistrationNumber'].dropna())
regs_maint     = set(maintenance['Registration Number'].dropna())

print('Registration number counts per dataset:')
print(f'  Vehicle_Info:         {len(regs_info):,}')
print(f'  Vehicle_Driver:       {len(regs_driver):,}')
print(f'  Vehicle_Mileage:      {len(regs_mileage):,}')
print(f'  Maintenance_Records:  {len(regs_maint):,}')
print()

# Overlap
print('Overlap with Vehicle_Info (our master):')
print(f'  Driver in Info:    {len(regs_driver & regs_info):,}')
print(f'  Mileage in Info:   {len(regs_mileage & regs_info):,}')
print(f'  Maint in Info:     {len(regs_maint & regs_info):,}')
print()

print('In Maintenance but NOT in Vehicle_Info:')
maint_not_in_info = regs_maint - regs_info
print(f'  {len(maint_not_in_info)} registrations — these may be cars or disposed vehicles')

In [ ]:
# Vans with maintenance records
van_regs = set(vehicle_info[vehicle_info['Asset Type'].isin(van_types)]['RegistrationNumber'])
vans_with_maint = van_regs & regs_maint
vans_no_maint   = van_regs - regs_maint

print(f'Total van registrations: {len(van_regs):,}')
print(f'Vans with maintenance records: {len(vans_with_maint):,}')
print(f'Vans with NO maintenance records: {len(vans_no_maint):,}')
print('>> Vans with no records may be very new or recently acquired — worth reviewing in cleaning.')

## 10. Summary of Findings & Decisions

In [ ]:
print("""
========================================================
 EDA SUMMARY — KEY FINDINGS & DECISIONS FOR CLEANING
========================================================

FILTERING:
  [x] Keep only Asset Types: Medium Van, Small Van, Large Van
  [x] Disposed vehicles: include in model training, exclude from prediction output
  [x] Exclude 'Miscellaneous > Environmental Disposal' from repair modelling

MISSING VALUES:
  [x] Vehicle_Driver — missing Driver (92): assign fleet average driver score
  [x] Vehicle_Driver — missing Branch (57): fill from Vehicle_Info via registration
  [x] Vehicle_Mileage — non-numeric values in Average Miles Per Day / Year: coerce to NaN, flag

DATA TYPES:
  [x] Vehicle_Mileage: Average Miles Per Day and Year need numeric conversion
  [x] Maintenance_Records: date columns need datetime conversion
  [x] Driver_Scores: sub-score count columns stored as strings — convert to numeric

FEATURE DECISIONS:
  [x] Driver Score: aggregate weekly scores to a single average per driver
  [x] Vehicles with no driver: use fleet average score
  [x] MOT Date: retain as a feature — flag vehicles due within 90 days
  [x] Vehicle Age: derive from Registered Date in Vehicle_Info
  [x] Cumulative Mileage: sum Total Miles from Vehicle_Mileage per vehicle

MODELLING SCOPE:
  [x] Primary repair categories for survival analysis:
      Brakes, Wheels & Tyres, Engine, Electrical, Cooling
  [x] Driver score will use current assignment at time of model run
  [x] Future improvement: incorporate driver change timeline
========================================================
""")